# Training on the GPU cluster — jobs, hydra, and the numbers they produce

Companion to `notebooks/walkthrough.ipynb`. The walkthrough explains *the method* — the extended
coupling, the probability path, the Bregman objective. This notebook explains *the infrastructure*, and
then analyses what a run on the cluster actually produced:

1. **How a job becomes a configuration.** `sbatch` → `srun` → `train.py` → hydra's composed config, and
   where each override ends up. Worth understanding before changing anything.
2. **What is logged, and where.** The two log surfaces in a run directory, and how they differ.
3. **The analysis** — loss, and the precision/recall of the delete operation, per epoch.

### How to run it

| situation | what to do |
|---|---|
| laptop, artifacts pulled from the cluster | `git fetch origin && git restore --source origin/run-artifacts/<run> -- runs/` |
| on a cluster node via JupyterHub | choose the `Python (EditFlows_env)` kernel |
| a local CPU run | `python train.py experiment=deletion-smoke` first |

No cell needs a GPU, a checkpoint, or the training data — only the text files in a run directory. That
is deliberate: the analysis then always works on the laptop, even with no tunnel to the cluster. Point
it at a specific run with the `EDITFLOW_RUN_DIR` environment variable, or edit `CANDIDATES` below.

In [ ]:
import json, os, re, sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

ROOT = Path.cwd() if (Path.cwd() / "editflows").is_dir() else Path.cwd().parent
assert (ROOT / "editflows").is_dir(), "start the notebook inside the repository"
os.chdir(ROOT)                      # relative paths (runs/, config/) work like in train.py
sys.path.insert(0, str(ROOT))

# train.py registers this resolver at import time, before hydra composes. The configs use it for
# arithmetic such as ${eval:'25000 // ${data.batch_size}'}.
try:
    OmegaConf.register_new_resolver("eval", eval)
except ValueError:
    pass                            # already registered (re-running the cell)

# Okabe & Ito's qualitative palette: the standard colourblind-safe set. Assigned in a FIXED order, so a
# metric keeps its colour in every figure and colour never encodes rank.
CB = {"blue": "#0072B2", "vermillion": "#D55E00", "green": "#009E73", "orange": "#E69F00",
      "sky": "#56B4E9", "purple": "#CC79A7", "grey": "#7F7F7F"}
COLOR = {"del_precision": CB["blue"], "del_recall": CB["vermillion"], "del_f1": CB["green"],
         "del_chance": CB["grey"], "rank_p_at_k": CB["purple"],
         "train/loss": CB["blue"], "val/loss": CB["vermillion"],
         "tp": CB["green"], "tn": CB["sky"], "fp": CB["vermillion"], "fn": CB["orange"],
         "ins_per_seq": CB["orange"], "sub_per_seq": CB["purple"], "len_err": CB["blue"]}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "legend.frameon": False})

# ---- which run do we analyse? -----------------------------------------------------------------
CANDIDATES = [os.environ.get("EDITFLOW_RUN_DIR"),
              "runs/gpu-24h", "runs/gpu-check", "runs/cfgcheck",
              "runs/nb-demo", "runs/smoke", "runs/local"]
RUN_DIR = next((ROOT / c for c in CANDIDATES if c and (ROOT / c).is_dir()), None)

if RUN_DIR is None:
    print("No run directory found. Train something first, or pull one from the cluster:")
    print("  git fetch origin && git restore --source origin/run-artifacts/<run> -- runs/")
else:
    print(f"run directory: {RUN_DIR.relative_to(ROOT)}")
    runs_root = ROOT / "runs"
    if runs_root.is_dir():
        print("available:", ", ".join(sorted(p.name for p in runs_root.iterdir() if p.is_dir())))

In [ ]:
# Loaders used by every later cell. They return None rather than raising, so a run that is still
# training, has a single epoch, or arrived without checkpoints never breaks the notebook.

def note(msg):
    print(f"[skipped] {msg}")

def load_eval(run_dir):
    """eval/deletion_metrics.csv — one row per (stage, epoch, bucket), plus an `all` row per epoch."""
    if run_dir is None:
        return None
    p = run_dir / "eval" / "deletion_metrics.csv"
    return pd.read_csv(p) if p.is_file() else None

def overall(ev, stage="val"):
    """The per-epoch summary rows: bucket `all`, one row per epoch."""
    if ev is None:
        return None
    d = ev[(ev.target_len == "all") & (ev.stage == stage)].sort_values("epoch")
    return d if len(d) else None

def load_lightning(run_dir):
    """Concatenate logs/version_*/metrics.csv in version order — one directory per (re)start.

    Rows are NOT de-duplicated here: one step legitimately appears on several rows, because lightning
    writes step metrics and epoch-end metrics separately. Use `series()` to pull one metric out.
    """
    if run_dir is None or not (run_dir / "logs").is_dir():
        return None
    paths = sorted((run_dir / "logs").glob("version_*/metrics.csv"),
                   key=lambda p: int(re.search(r"version_(\d+)", p.parent.name).group(1)))
    frames = []
    for p in paths:
        try:
            f = pd.read_csv(p)
        except pd.errors.EmptyDataError:
            continue
        if len(f):
            frames.append(f)
    return pd.concat(frames, ignore_index=True) if frames else None

def series(df, col, x="step"):
    """One metric from the sparse lightning CSV: the rows where it is present, newest per step.

    `keep="last"` is what de-duplicates a resumed run: the later version_N was concatenated last, so
    its value for a repeated step wins.
    """
    if df is None or col not in df.columns:
        return None
    d = df[[x, col]].dropna()
    if not len(d):
        return None
    return d.drop_duplicates(subset=x, keep="last").sort_values(x)

EV = load_eval(RUN_DIR)
OV = overall(EV)
LOG = load_lightning(RUN_DIR)
CFG = (OmegaConf.load(RUN_DIR / "config.yaml")
       if RUN_DIR and (RUN_DIR / "config.yaml").is_file() else None)

print(f"eval/deletion_metrics.csv  : {'-' if EV is None else str(EV.shape[0]) + ' rows x ' + str(EV.shape[1]) + ' cols'}")
print(f"  of which `all` rows      : {'-' if OV is None else str(len(OV)) + ' epochs'}")
print(f"lightning CSV (all starts) : {'-' if LOG is None else str(LOG.shape[0]) + ' rows x ' + str(LOG.shape[1]) + ' cols'}")
print(f"resolved config.yaml       : {'-' if CFG is None else 'loaded'}")

---
## 1. From `sbatch` to a configuration

### 1.1 Config groups compose into one object

`config/train.yaml` is the entry point. Its `defaults` list names one file from each *config group*, and
`experiment=...` is applied **last**, so an experiment can override anything. Because the experiment
files carry `# @package _global_`, their keys land at the root of the config rather than under
`experiment.`.

The cell below composes the config for `experiment=deletion-3090-24h` **in this process** — the same
composition `train.py` performs, with no training and no GPU. This is the cheapest way to see what a job
would run before queueing it. On the command line the equivalent is
`python train.py experiment=deletion-3090-24h name=demo --cfg job --resolve`.

In [ ]:
from hydra import compose, initialize_config_dir

with initialize_config_dir(config_dir=str(ROOT / "config"), version_base=None):
    composed = compose(config_name="train",
                       overrides=["experiment=deletion-3090-24h", "name=demo"])

print(OmegaConf.to_yaml(composed.trainer))
print("encoder:", OmegaConf.to_container(composed.task.model.encoder, resolve=True))

Note what happened to `trainer`: `config/train.yaml` supplied `gradient_clip_val`, `max_steps: -1` and
`check_val_every_n_epoch`; `experiment/deletion-base.yaml` supplied `precision: bf16-mixed`; and
`experiment/deletion-3090-24h.yaml` supplied `max_epochs: 300` and the `limit_train_batches` expression.
Three files, one `trainer` node, last one wins.

The encoder printed above is `editflows.encoder.LlamaEncoder` — the **placeholder** sequence encoder, a
small Llama trunk with bidirectional attention inside each sequence. The training it drives is real; only
the encoder is a stand-in, and it is swapped by replacing the `encoder` node of
`config/task/model/encoder-head.yaml` with anything satisfying the `SequenceEncoder` interface.

### 1.2 Where an `sbatch` override ends up

```
sbatch scripts/train.sbatch long-01 task.optimizer.lr=1e-4
  │
  ├─ #SBATCH --gres=gpu:1 --cpus-per-task=8    SLURM sets SLURM_GPUS_ON_NODE=1, SLURM_CPUS_PER_TASK=8
  ├─ EDITFLOW_EXPERIMENT                       (default: deletion-3090-24h)
  │
  └─ srun python train.py \
         experiment=deletion-3090-24h \        ← from the environment, not from "$@"
         name=long-01 \                        → run_dir: runs/long-01
         gpus=1 \                              → trainer.devices: ${gpus}
         data.num_workers=7 \                  ← SLURM_CPUS_PER_TASK - 1
         task.optimizer.lr=1e-4                ← your "$@" passthrough
```

Two details worth internalising:

* **`gpus` is not a trainer setting.** It is a top-level key that `trainer.devices: ${gpus}`
  interpolates, and that `config/hydra/launcher/slurm.yaml` also reads for `gres: gpu:${gpus}`. One
  number drives both the SLURM request and the lightning trainer, so they cannot drift apart.
* **The experiment comes from an environment variable, not from `"$@"`.** Hydra refuses two values for
  one key, so a hard-coded `experiment=` in the script plus an `experiment=` in your arguments is an
  error, not an override. Use `EDITFLOW_EXPERIMENT=deletion-3090-check sbatch ...`.

The run directory keeps the receipt: `train.py` writes `host`, `process_id` and `slurm_job_id` into the
config it saves, so you can always prove which machine a run executed on.

In [ ]:
if CFG is None:
    note("no config.yaml in the run directory")
else:
    for k in ("host", "process_id", "slurm_job_id"):
        print(f"{k:12s} {CFG.get(k, '(absent - not a SLURM job)')}")
    print(f"{'run_dir':12s} {CFG.run_dir}")
    print(f"{'experiment':12s} {CFG.experiment.name}")

    # Hydra also keeps the raw command line, the most direct record of what was typed.
    ov = RUN_DIR / "hydra" / ".hydra" / "overrides.yaml"
    print(f"\noverrides.yaml: {OmegaConf.load(ov) if ov.is_file() else '(absent)'}")

In [ ]:
# What did the experiment and the command line actually change, relative to the bare defaults?
def flat(cfg):
    out = {}
    container = cfg if isinstance(cfg, dict) else OmegaConf.to_container(cfg, resolve=True)
    def walk(node, path):
        if isinstance(node, dict):
            for k, v in node.items():
                walk(v, f"{path}.{k}" if path else str(k))
        else:
            out[path] = node
    walk(container, "")
    return out

if CFG is None:
    note("no config.yaml to compare against")
else:
    with initialize_config_dir(config_dir=str(ROOT / "config"), version_base=None):
        bare = compose(config_name="train", overrides=[f"name={CFG.name}"])
    a, b = flat(bare), flat(CFG)
    noise = {"host", "process_id", "slurm_job_id"}   # bookkeeping, not configuration
    rows = [(k, a.get(k, "(new)"), b[k]) for k in b
            if k not in noise and (k not in a or a[k] != b[k])]
    print(f"{len(rows)} keys differ from the bare defaults:\n")
    for k, was, now in sorted(rows):
        print(f"  {k:46s} {str(was):>20s}  ->  {now}")

### 1.3 The `${eval:}` resolver, and what one epoch means here

There is no real dataset yet: the targets are synthetic, so "one pass over the training split" is an
arbitrary unit. The project therefore *defines* an epoch as **25 000 training sequences**, and writes it
in `config/experiment/deletion-3090-24h.yaml` as

```yaml
limit_train_batches: ${eval:'25000 // ${data.batch_size}'}
```

`train.py` registers `eval` as an OmegaConf resolver, so the config can do arithmetic. Expressing the
epoch this way rather than as a literal `195` means the definition survives a change of batch size —
which matters, because `data.batch_size` is the first knob you reach for when tuning GPU memory.

This is **not** truncation to the same 195 batches every epoch. The training dataloader uses a shuffling
`DistributedSampler`, and lightning calls `set_epoch` on it at the start of every epoch (`fit_loop.py`,
`on_advance_start`) regardless of `use_distributed_sampler`, so each epoch draws a *different* random
25 000 sequences from the much larger pool. Note also that `//` floors, so the true count is slightly
below 25 000 — 195 × 128 = 24 960.

In [ ]:
expr = "${eval:'25000 // ${data.batch_size}'}"
print("the invariant:")
for bs in (16, 32, 64, 128, 256):
    c = OmegaConf.create({"data": {"batch_size": bs}, "trainer": {"limit_train_batches": expr}})
    n = c.trainer.limit_train_batches
    print(f"  batch_size {bs:4d}  ->  {n:5d} batches  ->  {n * bs:6d} sequences per epoch")

if CFG is None:
    note("no config.yaml - cannot check the run itself")
else:
    bs = CFG.data.batch_size
    lim = CFG.trainer.get("limit_train_batches")
    tail = f"  ->  {lim * bs} sequences per epoch" if isinstance(lim, int) else "  (unlimited)"
    print(f"\nthis run: batch_size={bs}, limit_train_batches={lim}{tail}")

    # Cross-check against what the logs observed, rather than trusting the config alone.
    if OV is not None and len(OV) >= 2:
        gaps = sorted(set(OV.step.diff().dropna().astype(int)))
        print(f"observed steps between validation epochs: {gaps}   (should equal limit_train_batches)")
    else:
        note("fewer than two validation epochs - nothing to cross-check yet")

### 1.4 The run directory, and why a 24-hour job is restartable

Everything a run writes goes to `runs/<name>/`, and **the run name is the identity of the run**. Starting
the same name again does not start over: `train.py`'s `resume_path()` picks the newest of
`checkpoints/last.ckpt` and `checkpoints/periodic.ckpt` *by modification time* and resumes from it.
`from_ckpt=<path>` overrides that; `from_ckpt=none` forces a fresh start.

That one rule is what makes the long run survive a time limit. `last.ckpt` is written after every
validation epoch and `periodic.ckpt` every `checkpoint.time_interval_min` minutes (30 by default), so a
job killed by SLURM loses at most half an hour — and `scripts/submit_chain.sh N <name>` queues N
dependent jobs that each pick up where the previous one stopped.

Two consequences visible in the listing below:

* **One `logs/version_N/` per (re)start.** The per-step CSV is a sequence of files, not one file, which
  is why `load_lightning()` concatenates them in version order.
* **`eval/deletion_metrics.csv` is rewritten, not appended, from a resumed epoch onward**, so resuming
  never leaves two conflicting rows for one epoch.

In [ ]:
if RUN_DIR is None:
    note("no run directory")
else:
    rows = [(str(p.relative_to(RUN_DIR)).replace("\\", "/"), p.stat().st_size)
            for p in sorted(RUN_DIR.rglob("*")) if p.is_file() and ".hydra" not in p.parts]
    width = max((len(r[0]) for r in rows), default=10)
    for name, size in rows:
        kb = size / 1024
        flag = "   <- large; stays on the cluster" if kb > 5000 else ""
        print(f"  {name:<{width}}  {kb:9.1f} KiB{flag}")

    ck = RUN_DIR / "checkpoints"
    cands = [p for p in (ck / "last.ckpt", ck / "periodic.ckpt") if p.is_file()] if ck.is_dir() else []
    if cands:
        newest = max(cands, key=lambda p: p.stat().st_mtime)
        print(f"\nresume_path() would choose: {newest.name}")
        for p in cands:
            print(f"   {p.name:16s} {datetime.fromtimestamp(p.stat().st_mtime):%Y-%m-%d %H:%M:%S}")
    else:
        note("no checkpoints/ - artifacts pulled from the cluster without them (expected)")

### 1.5 SLURM, and two traps

`train.py` installs `SLURMEnvironment(auto_requeue=False)` when it detects SLURM. Lightning's built-in
auto-requeue would write a checkpoint *from inside a signal handler*, which can fire mid-step and is
fragile under DDP; resubmitting a job that resumes from a periodic checkpoint is simpler and does not
depend on signal timing. A deliberate choice, not an oversight.

Two things that will bite you if you edit the configs:

1. **`trainer.max_time` does not survive a resume.** Lightning's `Timer` checkpoints its elapsed time and
   restores it cumulatively, so a run bounded at 28 minutes stops almost immediately the second time it
   starts. That is why neither GPU experiment sets it, and why the ~30-minute run is
   `trainer.max_epochs=8` instead.
2. **The cosine horizon is computed once, from `max_epochs`.** `DeletionTask.configure_optimizers`
   replaces a non-positive `max_steps` with `trainer.estimated_stepping_batches`, i.e.
   `max_epochs × limit_train_batches`. Stop a run early and the learning rate simply never finishes
   decaying; *change* `max_epochs` on a resumed run and the schedule shifts under you.

If `neuroai` turns out not to be managed by SLURM, none of this changes: `resume_path()` is
checkpoint-based, not SLURM-based, so a detached `python train.py … name=<same name>` resumes identically.

### 🧑‍💻 Exercise 1

Without training anything, work out how many optimizer steps the ~30-minute run performs, and what
cosine horizon its scheduler is handed. Then judge whether `warmup_steps` is a sensible fraction of it.

*Hint:* compose `experiment=deletion-3090-24h` with the override `trainer.max_epochs=8`, then look at
`trainer.limit_train_batches` and `task.lr_scheduler.warmup_steps`.

In [ ]:
# 🧑‍💻 Exercise 1 — your code here

---
## 2. What is logged, and where

A run writes its numbers to **two** places, and they are not interchangeable:

| file | granularity | written by | use it for |
|---|---|---|---|
| `logs/version_N/metrics.csv` | per **step** (and epoch ends) | lightning's `CSVLogger` | the within-epoch loss trace, the learning rate |
| `eval/deletion_metrics.csv` | per **epoch × bucket** | `editflows.metrics.write_metrics` | everything per-epoch; raw counts *and* rates |

The second is the authoritative per-epoch source, and the one to prefer: it carries the raw confusion
counts, not just the derived rates, and it has one tidy row per epoch rather than a sparse wide table.

In [ ]:
if OV is None:
    note("no per-epoch metrics yet")
else:
    cols = ["epoch", "step", "n_seq", "tp", "fp", "fn", "tn",
            "del_precision", "del_recall", "del_f1", "del_chance", "rank_p_at_k"]
    print(f"last {min(6, len(OV))} validation epochs (bucket `all`):\n")
    print(OV[cols].tail(6).to_string(index=False))

### 2.1 What each column means

The task is deletion-only: the source is a known target with `K` uniform letters inserted at random
positions, and the model must remove exactly those. So the confusion matrix is defined **per source
token**, on whether it was deleted, against whether it had been inserted
(`editflows/metrics.py`, `DeletionCounts.update`):

| | token was **inserted** | token was **original** |
|---|---|---|
| model **deleted** it | `tp` | `fp` |
| model **kept** it | `fn` | `tn` |

| column | meaning |
|---|---|
| `del_precision` | `tp / (tp + fp)` — of the tokens it deleted, how many should have gone |
| `del_recall` | `tp / (tp + fn)` — of the inserted tokens, how many it found |
| `del_f1` | their harmonic mean; this is what `checkpoint.monitor` selects the best checkpoint on |
| `del_chance` | `(tp + fn) / (tp + fp + fn + tn)` — the share of inserted tokens, i.e. the precision of deleting *at random*. **The baseline every precision must be read against.** |
| `rank_p_at_k` | threshold-free: at `t = 0`, the share of insertions among the `k` tokens with the highest deletion rate, `k` = number of insertions. Moves before precision does. |
| `ins_per_seq`, `sub_per_seq` | spurious insertion / substitution events per sequence. The data never needs either; the target is 0. |
| `exact_match`, `len_err` | sequence level: sample equals the target; mean absolute length error |

**`del_precision` alone tells you almost nothing.** A model that deletes tokens at random scores
`del_precision ≈ del_chance ≈ 0.4` on this data. Only the *gap* between them is learning. Let us verify
the rates really are the counts, and check the gap:

In [ ]:
if OV is None:
    note("no per-epoch metrics yet")
else:
    last = OV.iloc[-1]
    tp, fp, fn, tn = (float(last[k]) for k in ("tp", "fp", "fn", "tn"))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    chance = (tp + fn) / (tp + fp + fn + tn) if tp + fp + fn + tn else 0.0

    print(f"epoch {int(last.epoch)}:  tp={tp:.0f} fp={fp:.0f} fn={fn:.0f} tn={tn:.0f}\n")
    for name, mine in (("del_precision", prec), ("del_recall", rec),
                       ("del_f1", f1), ("del_chance", chance)):
        logged = float(last[name])
        ok = "ok" if abs(mine - logged) < 1e-6 else "MISMATCH"
        print(f"  {name:14s} recomputed {mine:.6f}   logged {logged:.6f}   {ok}")

    lift = prec - chance
    verdict = ("at chance - it has not learned WHICH tokens to delete yet"
               if lift < 0.02 else f"{lift:+.3f} above chance")
    print(f"\nprecision - chance = {lift:+.4f}   ->   {verdict}")

### 2.2 One honest gap: no precision/recall for insert or substitute

The model emits three operations — insert (`log_lambdas[0]`), substitute (`[1]`) and delete (`[2]`) — but
only delete gets a confusion matrix. That is not an oversight. Under the extended coupling the source is
*only* ever longer than the target, so the correct edit script is pure deletions: there is no positive
class for insert or substitute to recall. Any insertion or substitution the model emits is a false
positive by construction, which is why those two are counted as **event rates**
(`ins_per_seq`, `sub_per_seq`) whose target is zero, rather than as precision/recall.

If the coupling ever gains real insertions or substitutions, that changes, and the counts in
`editflows/metrics.py` would need a positive class per operation.

### 2.3 The lightning CSV is sparse, and that is a trap

`CSVLogger` writes one row per logging event and unions all the columns, so most cells are `NaN`:
`train/loss` appears on step rows, every `val/*` column only on epoch-boundary rows. Plotting a column
without dropping the `NaN`s gives an empty or broken line, which is exactly why `series()` exists.

In [ ]:
if LOG is None:
    note("no lightning CSV in this run")
else:
    interesting = [c for c in ("step", "epoch", "train/loss", "lr-AdamW", "val/loss",
                               "val/del_precision", "val/del_recall") if c in LOG.columns]
    head = LOG[interesting].head(12)
    print(f"{LOG.shape[0]} rows x {LOG.shape[1]} columns; first rows of a few columns:\n")
    print(head.to_string(index=False))

    print("\nnon-null count per column:")
    for c in interesting:
        print(f"  {c:20s} {LOG[c].notna().sum():5d} / {len(LOG)}")

    versions = sorted((RUN_DIR / "logs").glob("version_*"))
    plural = "" if len(versions) == 1 else "s"
    print(f"\n{len(versions)} version directory{'' if len(versions) == 1 else 'ies'} "
          f"= {len(versions)} (re)start{plural}: {', '.join(p.name for p in versions)}")
    tl = series(LOG, "train/loss")
    if tl is not None:
        print(f"series(LOG, 'train/loss') -> {len(tl)} usable points, "
              f"steps {int(tl.step.min())}..{int(tl.step.max())}")

### 🧑‍💻 Exercise 2

`load_lightning()` deliberately does *not* de-duplicate rows, while `series()` does, with
`keep="last"`. Construct the situation where that matters — two `version_*` directories covering an
overlapping range of steps — and convince yourself which value survives and why that is the right one.

*Hint:* a resumed run re-reports the steps between the last checkpoint and where it was killed.

In [ ]:
# 🧑‍💻 Exercise 2 — your code here

---
## 3. The run itself

Everything below degrades gracefully: with one epoch you get points instead of lines, and cells whose
input is missing say so and move on.

### 3.1 Loss

In [ ]:
tl = series(LOG, "train/loss")
vl = series(LOG, "val/loss")

if tl is None:
    note("no train/loss logged yet")
else:
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(tl.step, tl["train/loss"], color=COLOR["train/loss"], lw=2, label="train/loss",
            marker="o" if len(tl) < 25 else None, ms=4)
    if vl is not None:
        ax.plot(vl.step, vl["val/loss"], color=COLOR["val/loss"], lw=2, ls="--",
                marker="o", ms=5, label="val/loss")
    # Epoch boundaries, so the within-epoch trace is readable.
    if CFG is not None and isinstance(CFG.trainer.get("limit_train_batches"), int):
        per = CFG.trainer.limit_train_batches
        for b in range(per, int(tl.step.max()) + 1, per):
            ax.axvline(b, color=CB["grey"], lw=0.8, alpha=0.35)
    ax.set_yscale("log")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("Bregman divergence")
    ax.set_title("Training loss" + ("" if vl is None else " and validation loss")
                 + "   (grey lines: epoch boundaries)")
    ax.legend()
    plt.tight_layout(); plt.show()

### 3.2 The delete operation, per epoch

This is the figure the whole cluster setup exists to produce: precision, recall and F1 of the delete
operation after every epoch, against the chance baseline. The dashed grey line is `del_chance` — a model
deleting at random sits exactly on it, so **only the vertical distance above it is learning.**

In [ ]:
if OV is None:
    note("no per-epoch deletion metrics yet - the first validation epoch has not finished")
else:
    single = len(OV) == 1
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

    ax = axes[0]
    for m in ("del_precision", "del_recall", "del_f1"):
        ax.plot(OV.epoch, OV[m], color=COLOR[m], lw=2, marker="o", ms=5 if single else 4, label=m)
    ax.plot(OV.epoch, OV.del_chance, color=COLOR["del_chance"], lw=1.5, ls="--",
            label="del_chance (random deletion)")
    ax.set_xlabel("epoch"); ax.set_ylabel("rate"); ax.set_ylim(0, 1)
    ax.set_title("Delete operation per epoch")
    ax.legend(loc="best", fontsize=8)

    ax = axes[1]
    ax.plot(OV.epoch, OV.del_precision - OV.del_chance, color=COLOR["del_precision"], lw=2,
            marker="o", ms=4, label="del_precision - del_chance")
    ax.plot(OV.epoch, OV.rank_p_at_k - OV.del_chance, color=COLOR["rank_p_at_k"], lw=2,
            marker="s", ms=4, label="rank_p_at_k - del_chance")
    ax.axhline(0, color=COLOR["del_chance"], lw=1.5, ls="--")
    ax.set_xlabel("epoch"); ax.set_ylabel("lift over chance")
    ax.set_title("Is it actually learning?")
    ax.legend(loc="best", fontsize=8)

    plt.tight_layout(); plt.show()

    if single:
        print("one epoch only - the trend needs at least two.")
    best = OV.loc[OV.del_f1.idxmax()]
    print(f"best del_f1 {best.del_f1:.4f} at epoch {int(best.epoch)}  "
          f"(precision {best.del_precision:.4f}, recall {best.del_recall:.4f}, "
          f"chance {best.del_chance:.4f})")
    print(f"checkpoint.monitor = {CFG.checkpoint.monitor if CFG is not None else 'val/del_f1'} "
          f"-> this is the epoch best-*.ckpt was saved from")

`rank_p_at_k` is the more sensitive of the two curves on the right. It asks only whether the
model *ranks* the inserted tokens above the original ones by deletion rate at `t = 0`, so it moves while
precision is still pinned at chance — useful early evidence that the encoder is learning something
before the sampler turns it into actual deletions.

### 3.3 The learning-rate schedule

In [ ]:
lr = series(LOG, "lr-AdamW")
if lr is None:
    note("no lr-AdamW column (LearningRateMonitor logs it once training starts)")
else:
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.plot(lr.step, lr["lr-AdamW"], color=CB["blue"], lw=2)
    warm = None
    if CFG is not None:
        warm = CFG.task.lr_scheduler.get("warmup_steps")
    if warm:
        ax.axvline(warm, color=CB["grey"], lw=1.2, ls="--")
        ax.annotate(f"warmup ends ({warm})", xy=(warm, lr["lr-AdamW"].max() * 0.6),
                    xytext=(6, 0), textcoords="offset points", fontsize=8, color=CB["grey"])
    ax.set_xlabel("optimizer step"); ax.set_ylabel("learning rate")
    ax.set_title("Warmup then cosine annealing")
    plt.tight_layout(); plt.show()

    if CFG is not None and isinstance(CFG.trainer.get("limit_train_batches"), int):
        horizon = CFG.trainer.limit_train_batches * CFG.trainer.max_epochs
        print(f"cosine horizon = limit_train_batches x max_epochs = "
              f"{CFG.trainer.limit_train_batches} x {CFG.trainer.max_epochs} = {horizon} steps")
        print(f"reached so far: {int(lr.step.max())} steps "
              f"({100 * int(lr.step.max()) / horizon:.1f}% of the schedule)")

### 3.4 Where it works and where it does not

`eval/deletion_metrics.csv` buckets every epoch by target length and by original fraction
`f = n_1 / n_0` (the share of the source that is real). Low `f` means heavily corrupted and therefore
*easier* — there is more to delete and the chance baseline is higher. Buckets with `n_seq == 0` are left
blank rather than drawn as zero: with a small `task.eval.n_sample_seqs` most buckets are simply empty,
and plotting them as 0 invents a result that was never measured.

In [ ]:
if EV is None:
    note("no bucket metrics yet")
else:
    buckets = EV[(EV.stage == "val") & (EV.target_len != "all")]
    buckets = buckets[buckets.epoch == buckets.epoch.max()]
    if not len(buckets) or not (buckets.n_seq > 0).any():
        note(f"every bucket is empty at epoch {int(EV.epoch.max())} "
             f"(task.eval.n_sample_seqs too small to fill them) - the `all` row carries the signal")
    else:
        def lo(s):
            return float(str(s).split("-")[0])
        rows = sorted(buckets.target_len.unique(), key=lo)
        cols = sorted(buckets.fraction.unique(), key=lo)
        f1 = buckets.pivot_table(index="target_len", columns="fraction",
                                 values="del_f1").reindex(index=rows, columns=cols)
        n = buckets.pivot_table(index="target_len", columns="fraction",
                                values="n_seq").reindex(index=rows, columns=cols)
        f1 = f1.mask(n.fillna(0) == 0)                     # never draw an unmeasured bucket

        fig, ax = plt.subplots(figsize=(1.5 + 1.3 * len(cols), 1.2 + 0.75 * len(rows)))
        im = ax.imshow(f1.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(cols)), cols); ax.set_yticks(range(len(rows)), rows)
        ax.set_xlabel("original fraction  n1/n0"); ax.set_ylabel("target length")
        ax.set_title(f"del_f1 per bucket, epoch {int(buckets.epoch.max())}")
        ax.grid(False)
        for i in range(len(rows)):
            for j in range(len(cols)):
                v, cnt = f1.values[i, j], n.values[i, j]
                label = "-" if np.isnan(v) else f"{v:.2f}\nn={cnt:.0f}"
                shade = "white" if (not np.isnan(v) and v > 0.6) else "black"
                ax.text(j, i, label, ha="center", va="center", fontsize=7, color=shade)
        fig.colorbar(im, ax=ax, label="del_f1", shrink=0.85)
        plt.tight_layout(); plt.show()

### 3.5 Is it deleting too much, or too little?

The rates hide the asymmetry; the raw counts do not. `fp` dominating means it is over-deleting (eating
original tokens), `fn` dominating means it is too timid. The counts are normalised per epoch because
`n_seq` and the sequence lengths vary between epochs.

In [ ]:
if OV is None or len(OV) < 1:
    note("no per-epoch counts yet")
else:
    parts = ["tp", "fp", "fn", "tn"]
    total = OV[parts].sum(axis=1).replace(0, np.nan)
    fig, ax = plt.subplots(figsize=(8, 3.2))
    bottom = np.zeros(len(OV))
    for p in parts:
        share = (OV[p] / total).fillna(0).values
        ax.bar(OV.epoch, share, bottom=bottom, color=COLOR[p], label=p,
               width=0.8, linewidth=0.8, edgecolor="white")
        bottom += share
    ax.set_xlabel("epoch"); ax.set_ylabel("share of source tokens"); ax.set_ylim(0, 1)
    ax.set_title("Where every source token ended up")
    ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22))
    ax.grid(axis="x", visible=False)
    plt.tight_layout(); plt.show()

    last = OV.iloc[-1]
    if last.fp > last.fn:
        print(f"epoch {int(last.epoch)}: fp ({last.fp:.0f}) > fn ({last.fn:.0f}) - over-deleting, "
              f"it is removing original tokens")
    elif last.fn > last.fp:
        print(f"epoch {int(last.epoch)}: fn ({last.fn:.0f}) > fp ({last.fp:.0f}) - under-deleting, "
              f"it is leaving insertions in place")

### 3.6 Operations the data never needs

The model is free to insert and substitute, but the correct edit script never does. These should fall to
zero; `len_err` is on a different scale, so it gets its own axis rather than being forced onto a shared
one.

In [ ]:
if OV is None:
    note("no per-epoch metrics yet")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
    ax = axes[0]
    for m in ("ins_per_seq", "sub_per_seq"):
        ax.plot(OV.epoch, OV[m], color=COLOR[m], lw=2, marker="o", ms=4, label=m)
    ax.axhline(0, color=CB["grey"], lw=1, ls="--")
    ax.set_xlabel("epoch"); ax.set_ylabel("events per sequence")
    ax.set_title("Spurious operations (target: 0)"); ax.legend(fontsize=8)

    ax = axes[1]
    ax.plot(OV.epoch, OV.len_err, color=COLOR["len_err"], lw=2, marker="o", ms=4)
    ax.set_xlabel("epoch"); ax.set_ylabel("mean |length error| (tokens)")
    ax.set_title("How far off is the output length?")
    plt.tight_layout(); plt.show()

    if "exact_match" in OV.columns:
        print(f"exact_match, last epoch: {OV.iloc[-1].exact_match:.4f} "
              f"(fraction of samples equal to the target)")

### 3.7 Calibrating the next job

The plumbing-check run exists to measure one number: how long an epoch takes on the real GPU. With that,
`max_epochs` for a 30-minute or a 24-hour job is arithmetic rather than a guess. The timestamps come from
the `val epoch N step S:` lines that `DeletionTask._report` writes to `train.log` after every validation
epoch.

Note the units: an "epoch" here is one training epoch **plus** its sampling evaluation, which is what the
wall clock actually sees. The evaluation can easily be a third of it — it runs `sample_steps` sequential
forward passes over `n_sample_seqs` sequences — so if an epoch is slower than you expect, suspect
`task.eval` before the model.

In [ ]:
LINE = re.compile(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}),\d+\]\[editflows\.deletion_task\]"
                  r"\[INFO\] (val|test) epoch (\d+) step (\d+):")

log_path = RUN_DIR / "train.log" if RUN_DIR else None
stamps = []
if log_path is not None and log_path.is_file():
    for line in log_path.read_text(encoding="utf-8", errors="replace").splitlines():
        m = LINE.match(line)
        if m and m.group(2) == "val":
            stamps.append((datetime.strptime(m.group(1), "%Y-%m-%d %H:%M:%S"), int(m.group(3))))

if not log_path or not log_path.is_file():
    note("no train.log (artifacts may have been pulled without it)")
elif len(stamps) < 2:
    note(f"only {len(stamps)} validation epoch(s) in train.log - two are needed for a rate")
else:
    # Consecutive epochs only: a resumed run leaves a gap where the job was queued.
    deltas = [(b[0] - a[0]).total_seconds()
              for a, b in zip(stamps, stamps[1:]) if b[1] == a[1] + 1]
    if not deltas:
        note("no two consecutive epochs in one job - cannot measure an epoch time")
    else:
        med = float(np.median(deltas))
        print(f"{len(deltas)} consecutive epoch interval(s); median {med:.1f} s "
              f"({med / 60:.2f} min), min {min(deltas):.0f} s, max {max(deltas):.0f} s")

        per_epoch, on_cpu = None, False
        if CFG is not None:
            lim = CFG.trainer.get("limit_train_batches")
            if isinstance(lim, int):
                per_epoch = lim * CFG.data.batch_size
            on_cpu = str(CFG.trainer.accelerator) == "cpu"
            print(f"measured on {CFG.get('host', '?')} with "
                  f"accelerator={CFG.trainer.accelerator}, precision={CFG.trainer.precision}, "
                  f"batch_size={CFG.data.batch_size}, "
                  f"{per_epoch if per_epoch else '?'} sequences/epoch, "
                  f"eval {CFG.task.eval.n_sample_seqs} seqs x {CFG.task.eval.sample_steps} steps")

        if on_cpu:
            print("\n!! this run was on the CPU - the numbers below do NOT transfer to a GPU job.")
            print("   Measure on the node: EDITFLOW_EXPERIMENT=deletion-3090-check "
                  "sbatch scripts/train.sbatch gpu-check")
        print("\nmax_epochs that fits a given wall clock (10% head-room for startup/checkpoints),")
        print("valid only for THIS run's model, batch size and task.eval settings:")
        for label, hours in (("30 min", 0.5), ("4 h", 4), ("23.5 h", 23.5)):
            n = max(int(hours * 3600 * 0.9 / med), 1)
            seen = f"   ({n * per_epoch:,} sequences seen)" if per_epoch else ""
            print(f"  {label:>7s}  ->  trainer.max_epochs={n:<6d}{seen}")
        print("\n  sbatch scripts/train.sbatch long-01 trainer.max_epochs=<N>")

### 🧑‍💻 Exercise 3

The numbers above were measured with this run's `task.eval` settings. Work out how much of an epoch is
the sampling evaluation rather than training, and what `n_sample_seqs` / `sample_steps` you would choose
for the 24-hour run to keep the evaluation under a tenth of the epoch.

*Hint:* the evaluation costs roughly `ceil(n_sample_seqs / eval_batch_size) × sample_steps` forward
passes; a training epoch costs `limit_train_batches` forward+backward passes. Lowering `sample_steps`
changes what `del_recall` *means* (fewer Euler steps, less opportunity to delete), so it is not
comparable across runs — `n_sample_seqs` only makes the estimate noisier.

---
## 4. Summary

* A run is identified by `name`; everything it writes is under `runs/<name>/`, and starting the same name
  again **resumes** it. That is the whole restart story for a 24-hour job.
* `experiment=` is applied last and can override anything; `gpus` is the one number that drives both the
  SLURM request and `trainer.devices`. The resolved `runs/<name>/config.yaml` is the ground truth of what
  ran, including the host and SLURM job id.
* One epoch is **defined** as 25 000 synthetic sequences, via
  `limit_train_batches: ${eval:'25000 // ${data.batch_size}'}`, and each epoch draws a different random
  subset.
* `del_precision` is meaningless without `del_chance` beside it. `rank_p_at_k` moves first.
* `trainer.max_time` is incompatible with resuming; bound long runs with `max_epochs` and the SLURM time
  limit.

### Where to go next

```bash
# the full picture for one checkpoint, with annotated examples
python evaluate.py runs/<name>/checkpoints/last.ckpt -n 512 --show 4 --out eval.json

# more wall clock for the same run - it continues from its newest checkpoint
sbatch scripts/train.sbatch <name>
scripts/submit_chain.sh 3 <name>

# bring the numbers home
scripts/push_run_artifacts.sh <name>
```

`notebooks/walkthrough.ipynb` covers the model and the objective; `docs/CLUSTER.md` covers submission and
the cluster runbook.